#1. Import required libraries and dataset

In [150]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import random
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from tabulate import tabulate

In [151]:
df = pd.read_csv('data.csv')
df = df.drop(['ap_lo', 'pulse_pressure'], axis = 1)
df

,gender,height,weight,ap_hi,cholesterol,gluc,smoke,alco,active,cardio,age_years,bp_category_encoded
0,1,0.463125,-0.873002,-1.043394,1,1,0,0,1,0,-0.412905,2
1,0,-1.107092,0.933963,0.965018,3,1,0,0,1,1,0.325344,3
2,0,0.070571,-0.715875,0.295547,3,1,0,0,0,1,-0.265255,2
3,1,0.593977,0.698272,1.634488,1,1,0,0,1,1,-0.708204,3
4,0,-1.107092,-1.344384,-1.712865,1,1,0,0,0,0,-0.855854,0
...,...,...,...,...,...,...,...,...,...,...,...,...
65681,0,0.986531,-0.244493,0.295547,1,1,0,0,1,1,0.030044,2
65682,0,0.070571,0.541144,1.634488,1,1,0,0,1,1,0.620643,2
65683,1,0.463125,0.226890,-0.373924,1,1,1,0,1,0,-0.117605,2
65684,0,-0.191132,-0.087365,0.630282,1,2,0,0,0,1,1.211242,2


In [152]:
SEED = 4240
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#2. Separate features X and target y and define functions for metrics and threshold search

In [153]:
X = df.drop('cardio', axis = 1)
y = df['cardio']

In [154]:
def get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred):
    table_data = [
        ['Train', precision_score(y_train, y_train_pred), recall_score(y_train, y_train_pred), f1_score(y_train, y_train_pred)],
        ['Validation', precision_score(y_val, y_val_pred), recall_score(y_val, y_val_pred), f1_score(y_val, y_val_pred)],
        ['Test', precision_score(y_test, y_test_pred), recall_score(y_test, y_test_pred), f1_score(y_test, y_test_pred)]
    ]
    print(tabulate(table_data, headers = ['Set', 'Precision', 'Recall', 'F1-Score'], tablefmt = 'grid'))

In [155]:
def threshold_search(threshold_range, X, y, model):
    best_precision = 0
    best_recall = 0
    best_f1 = 0
    best_threshold = 0
    for threshold in threshold_range:
        y_pred = (model.predict_proba(X)[:, 1] >= threshold).astype(int)
        precision = precision_score(y, y_pred)
        recall = recall_score(y, y_pred)
        f1 = f1_score(y, y_pred)
        if (precision >= 0.7 and recall >= 0.7 and f1 > best_f1):
            best_precision = precision
            best_recall = recall
            best_f1 = f1
            best_threshold = threshold
    return best_threshold

# 3. Train basic model

In [156]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.4, stratify = y, random_state = SEED)  # 60% train, 40% temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify = y_temp, random_state = SEED)  # 20% val, 20% test

In [157]:
mlp = MLPClassifier(hidden_layer_sizes = (64, 32), learning_rate_init = 0.001, alpha = 0, max_iter = 30, activation = 'relu', solver = 'adam', random_state = SEED, batch_size = 64)
mlp.fit(X_train, y_train)

/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(alpha=0, batch_size=64, hidden_layer_sizes=(64, 32), max_iter=30,
              random_state=4240)

In [158]:
y_train_pred = mlp.predict(X_train)
y_val_pred = mlp.predict(X_val)
y_test_pred = mlp.predict(X_test)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

+------------+-------------+----------+------------+
| Set        |   Precision |   Recall |   F1-Score |
+============+=============+==========+============+
| Train      |    0.728269 | 0.723464 |   0.725859 |
+------------+-------------+----------+------------+
| Validation |    0.720457 | 0.712647 |   0.716531 |
+------------+-------------+----------+------------+
| Test       |    0.722466 | 0.723374 |   0.72292  |
+------------+-------------+----------+------------+


In [ ]:
best_threshold = threshold_search(np.arange(0.3, 0.5, 0.01), X_val, y_val, mlp)
print(f'Best threshold: {best_threshold}')

y_train_pred = (mlp.predict_proba(X_train)[:, 1] >= best_threshold).astype(int)
y_val_pred = (mlp.predict_proba(X_val)[:, 1] >= best_threshold).astype(int)
y_test_pred = (mlp.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

# 4. Hyperparameter tuning with 5-fold cross validation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = SEED)  # 80% train, 20% test

In [ ]:
param_grid = {
    'alpha': [0, 0.001, 0.01],
    'learning_rate_init': [0.0001, 0.001, 0.01],
    'batch_size': [64, 128, 256]
}

mlp = MLPClassifier(random_state = SEED, max_iter = 100, hidden_layer_sizes = (64, 32), activation = 'relu', solver = 'adam')
grid_search = GridSearchCV(mlp, param_grid, cv = 5, scoring = 'f1', n_jobs = -1, verbose = 2)
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV] END ....alpha=0, batch_size=64, learning_rate_init=0.01; total time=   6.0s
[CV] END ....alpha=0, batch_size=64, learning_rate_init=0.01; total time=   9.0s
[CV] END ....alpha=0, batch_size=64, learning_rate_init=0.01; total time=   9.2s


/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END ....alpha=0, batch_size=64, learning_rate_init=0.01; total time=  10.5s


/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Use

[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  11.0s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  11.1s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  11.1s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  11.2s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  11.2s
[CV] END ...alpha=0, batch_size=64, learning_rate_init=0.001; total time=  11.2s
[CV] END ...alpha=0, batch_size=64, learning_rate_init=0.001; total time=  11.2s


...

/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.001; total time=   6.0s
[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.001; total time=   6.2s
[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.01; total time=   2.3s
[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.01; total time=   2.8s


/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.001; total time=   5.8s


GridSearchCV(cv=5,
             estimator=MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100,
                                     random_state=4240),
             n_jobs=-1,
             param_grid={'alpha': [0, 0.001, 0.01],
                         'batch_size': [64, 128, 256],
                         'learning_rate_init': [0.0001, 0.001, 0.01]},
             scoring='f1', verbose=2)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.4, stratify = y, random_state = SEED)  # 60% train, 40% temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify = y_temp, random_state = SEED)  # 20% val, 20% test


In [ ]:
best_model = grid_search.best_estimator_
best_parameters = grid_search.best_params_

print("Best Parameters:", best_parameters)

y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)
y_test_pred = best_model.predict(X_test)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

Best Parameters: {'alpha': 0.01, 'batch_size': 64, 'learning_rate_init': 0.0001}
+------------+-------------+----------+------------+
| Set        |   Precision |   Recall |   F1-Score |
+============+=============+==========+============+
| Train      |    0.7551   | 0.668744 |   0.709303 |
+------------+-------------+----------+------------+
| Validation |    0.748267 | 0.661273 |   0.702085 |
+------------+-------------+----------+------------+
| Test       |    0.756342 | 0.674364 |   0.713004 |
+------------+-------------+----------+------------+


In [ ]:
best_threshold = threshold_search(np.arange(0.3, 0.5, 0.01), X_val, y_val, best_model)
print(f'Best threshold: {best_threshold}')

y_train_pred = (best_model.predict_proba(X_train)[:, 1] >= best_threshold).astype(int)
y_val_pred = (best_model.predict_proba(X_val)[:, 1] >= best_threshold).astype(int)
y_test_pred = (best_model.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

Best threshold: 0.4100000000000001
+------------+-------------+----------+------------+
| Set        |   Precision |   Recall |   F1-Score |
+============+=============+==========+============+
| Train      |    0.704308 | 0.755092 |   0.728817 |
+------------+-------------+----------+------------+
| Validation |    0.704242 | 0.745954 |   0.724498 |
+------------+-------------+----------+------------+
| Test       |    0.706479 | 0.755419 |   0.73013  |
+------------+-------------+----------+------------+


# 5. Hyperparameter tuning with 10-fold cross validation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = SEED)  # 80% train, 20% test

In [ ]:
param_grid = {
    'alpha': [0, 0.001, 0.01],
    'learning_rate_init': [0.0001, 0.001, 0.01],
    'batch_size': [64, 128, 256]
}

mlp = MLPClassifier(random_state = SEED, max_iter = 100, hidden_layer_sizes = (64, 32), activation = 'relu', solver = 'adam')
grid_search = GridSearchCV(mlp, param_grid, cv = 10, scoring = 'f1', n_jobs = -1, verbose = 2)
grid_search.fit(X_train, y_train)

Fitting 10 folds for each of 27 candidates, totalling 270 fits
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=   8.7s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=   9.1s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=   9.9s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  11.8s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  12.3s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  12.5s


/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Use

[CV] END ...alpha=0, batch_size=64, learning_rate_init=0.001; total time=  12.6s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  12.7s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  12.7s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  12.8s
[CV] END ...alpha=0, batch_size=64, learning_rate_init=0.001; total time=  12.8s


/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END ...alpha=0, batch_size=64, learning_rate_init=0.001; total time=  12.9s
[CV] END ..alpha=0, batch_size=64, learning_rate_init=0.0001; total time=  13.0s


...

/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.001; total time=   7.7s
[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.01; total time=   3.8s
[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.001; total time=   7.4s
[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.001; total time=   7.3s


/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/timothy/Library/Python/3.10/lib/python/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.001; total time=   7.3s
[CV] END alpha=0.01, batch_size=256, learning_rate_init=0.01; total time=   3.9s


GridSearchCV(cv=10,
             estimator=MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100,
                                     random_state=4240),
             n_jobs=-1,
             param_grid={'alpha': [0, 0.001, 0.01],
                         'batch_size': [64, 128, 256],
                         'learning_rate_init': [0.0001, 0.001, 0.01]},
             scoring='f1', verbose=2)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.4, stratify = y, random_state = SEED)  # 60% train, 40% temp
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify = y_temp, random_state = SEED)  # 20% val, 20% test

In [ ]:
best_model = grid_search.best_estimator_
best_parameters = grid_search.best_params_

print("Best Parameters:", best_parameters)

y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)
y_test_pred = best_model.predict(X_test)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

Best Parameters: {'alpha': 0.001, 'batch_size': 128, 'learning_rate_init': 0.0001}
+------------+-------------+----------+------------+
| Set        |   Precision |   Recall |   F1-Score |
+============+=============+==========+============+
| Train      |    0.74714  | 0.680473 |   0.71225  |
+------------+-------------+----------+------------+
| Validation |    0.743377 | 0.67447  |   0.707249 |
+------------+-------------+----------+------------+
| Test       |    0.748627 | 0.68536  |   0.715598 |
+------------+-------------+----------+------------+


In [ ]:
best_threshold = threshold_search(np.arange(0.3, 0.5, 0.01), X_val, y_val, best_model)
print(f'Best threshold: {best_threshold}')

y_train_pred = (best_model.predict_proba(X_train)[:, 1] >= best_threshold).astype(int)
y_val_pred = (best_model.predict_proba(X_val)[:, 1] >= best_threshold).astype(int)
y_test_pred = (best_model.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)

get_metrics(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

Best threshold: 0.4200000000000001
+------------+-------------+----------+------------+
| Set        |   Precision |   Recall |   F1-Score |
+============+=============+==========+============+
| Train      |    0.704628 | 0.754255 |   0.728597 |
+------------+-------------+----------+------------+
| Validation |    0.704288 | 0.743126 |   0.723186 |
+------------+-------------+----------+------------+
| Test       |    0.707554 | 0.754791 |   0.73041  |
+------------+-------------+----------+------------+


# 10 Fold Cross Validation for T-test with LogReg 
refer to LogReg notebook for T-test results

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 4240)  # 80% train+val, 20% test

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Best MLP parameters
best_params = {
    'alpha': 0.001,
    'batch_size': 128,
    'learning_rate_init': 0.0001,
    'max_iter': 300,
    'hidden_layer_sizes': (64, 32), 
    'activation': 'relu',
    'solver': 'adam',
    'random_state': 4240    
}

threshold = 0.42
n_splits = 10

f1_scores = []
precision_scores = []
recall_scores = []

best_model = None
best_f1 = -1

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=4240)

fold = 1
for train_idx, val_idx in skf.split(X_train_val, y_train_val):
    X_train_fold = X_train_val.iloc[train_idx]
    y_train_fold = y_train_val.iloc[train_idx]
    X_val_fold = X_train_val.iloc[val_idx]
    y_val_fold = y_train_val.iloc[val_idx]

    model = MLPClassifier(**best_params)
    model.fit(X_train_fold, y_train_fold)

    y_probs = model.predict_proba(X_val_fold)[:, 1]
    y_pred = (y_probs >= threshold).astype(int)

    f1 = f1_score(y_val_fold, y_pred)
    precision = precision_score(y_val_fold, y_pred)
    recall = recall_score(y_val_fold, y_pred)

    f1_scores.append(f1)
    precision_scores.append(precision)
    recall_scores.append(recall)

    print(f"Fold {fold}: Precision = {precision:.4f}, Recall = {recall:.4f}, F1 Score = {f1:.4f}")

    if f1 > best_f1:
        best_model = model
        best_f1 = f1

    fold += 1

print("\n--- Average Scores over 10 folds ---")
print(f"Average Precision: {np.mean(precision_scores):.4f}")
print(f"Average Recall:    {np.mean(recall_scores):.4f}")
print(f"Average F1 Score:  {np.mean(f1_scores):.4f}")



Fold 1: Precision = 0.7124, Recall = 0.7617, F1 Score = 0.7362
Fold 2: Precision = 0.7018, Recall = 0.7550, F1 Score = 0.7274
Fold 3: Precision = 0.6975, Recall = 0.7435, F1 Score = 0.7198
Fold 4: Precision = 0.6955, Recall = 0.7698, F1 Score = 0.7308
Fold 5: Precision = 0.7059, Recall = 0.7459, F1 Score = 0.7254
Fold 6: Precision = 0.7079, Recall = 0.7643, F1 Score = 0.7350
Fold 7: Precision = 0.7003, Recall = 0.7765, F1 Score = 0.7364
Fold 8: Precision = 0.6955, Recall = 0.7455, F1 Score = 0.7196
Fold 9: Precision = 0.6989, Recall = 0.7494, F1 Score = 0.7233
Fold 10: Precision = 0.6855, Recall = 0.7730, F1 Score = 0.7266

--- Average Scores over 10 folds ---
Average Precision: 0.7001
Average Recall:    0.7585
Average F1 Score:  0.7281

--- Evaluating Best MLP Model on Test Set ---
Test Accuracy: 0.7186
Test Classification Report:
              precision    recall  f1-score   support

           0     0.7429    0.6943    0.7178      6772
           1     0.6960    0.7444    0.7194    